## What is PyTorch?

PyTorch is a Python-based deep learning library that runs on **CPU by default** and supports **GPU acceleration via CUDA**.

| Feature | What it means |
|---|---|
| Dynamic computation graphs | The graph is built *as your code runs*, not before |
| Automatic differentiation | Gradients are computed for you during backpropagation |
| CUDA support | Move tensors to GPU with `.to('cuda')` for faster training |

![PyTorch](pytorch.webp)

---

## <br>CUDA
CUDA (Compute Unified Device Architecture) is a GPU computing platform and programming model from NVIDIA that exposes hardware-level parallel execution capabilities to software.

- Provides direct access to GPU cores for general-purpose parallel computation.
- Executes kernels using thousands of concurrent threads organized into blocks and grids.
- Offloads data-parallel workloads from the CPU to the GPU for higher throughput.
- Commonly used in AI training, deep learning inference and high-performance computing workloads.

---

### <br>What is a Dynamic Computation Graph?

A **computation graph** is a record of every math operation applied to your tensors. PyTorch uses this graph to compute gradients (via backpropagation) when training a model.

**Dynamic** means the graph is built **on the fly, during each forward pass** — not compiled once upfront.

> Compare: TensorFlow 1.x used *static* graphs — you defined the entire graph first, then fed data in. Debugging was hard because the graph was a black box.

---

### Why Does It Matter?

With a dynamic graph you can use **normal Python control flow** inside your model — `if` statements, `for` loops, anything — and PyTorch adapts the graph each time.

**Use case: variable-length sequences**

Imagine a model that processes sentences. Sentences have different lengths, so you need a loop that runs a different number of times per input:

Each call builds a **different graph** — the short sentence gets a 3-step graph, the long sentence gets a 10-step graph. PyTorch handles both without any special configuration.

With a static graph this would require padding, masking, and pre-defining the max length upfront.

---

In [6]:
import torch
import torch.nn as nn

rnn = nn.RNNCell(input_size=10, hidden_size=20)

def process_sentence(words):          # words is a list of tensors, length varies
    hidden = torch.zeros(1, 20)
    for word in words:                # loop runs 3 times for short, 10 for long
        hidden = rnn(word, hidden)
    return hidden

short_sentence = [torch.randn(1, 10) for _ in range(3)]
long_sentence  = [torch.randn(1, 10) for _ in range(10)]

print("short_sentence: ", short_sentence)
print("long_sentence: ", long_sentence)

out1 = process_sentence(short_sentence)   # graph has 3 RNN steps
out2 = process_sentence(long_sentence)    # graph has 10 RNN steps

print("out1: ", out1)
print("out2: ", out2)

short_sentence:  [tensor([[-0.4320, -1.2528,  0.9625, -1.0896,  0.5333, -0.3441, -0.8556,  0.0226,
          2.5244,  2.4540]]), tensor([[-1.6628,  1.6715,  0.9093, -1.3427, -0.5542,  2.0131, -1.5683, -0.8366,
         -0.6461,  1.1405]]), tensor([[ 0.3022, -0.5465, -1.2380, -0.3095, -0.6925, -0.3595,  0.5498, -0.0127,
          1.1570, -0.1320]])]
long_sentence:  [tensor([[-0.9625, -1.0645, -1.3108, -0.9436,  0.3798,  0.0211, -1.6317, -0.4775,
         -0.8298, -1.4011]]), tensor([[ 0.6343, -0.5893,  0.9977,  0.8260,  0.6325,  1.6717, -0.9237,  0.7068,
          1.5015, -1.0544]]), tensor([[-1.5139,  0.4862,  0.0200, -1.0607, -1.4151, -0.5651, -0.4009, -0.3127,
          0.7318, -0.7430]]), tensor([[-1.2647,  0.0453, -0.4387,  1.9621, -1.0200,  0.8634,  0.7850, -0.6077,
         -0.0473, -2.0739]]), tensor([[ 1.1953,  0.2408, -0.4416, -0.5268, -0.6693,  0.3757,  2.9011, -0.2455,
          0.1099,  0.9960]]), tensor([[-0.2404,  0.0875,  0.4382, -0.5683, -1.0499,  0.7769,  1.2022,  0.70

### Installation

```bash
# CPU only
pip install torch torchvision torchaudio
```

## <br>PyTorch Tensors
Tensors are the fundamental data structures in PyTorch, similar to NumPy arrays but with GPU acceleration capabilities. PyTorch tensors support automatic differentiation, making them suitable for deep learning tasks.

In [7]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
print('1D Tensor: \n', x)

y = torch.zeros((3, 3))
print('2D Tensor: \n', y)

1D Tensor: 
 tensor([1., 2., 3.])
2D Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


## <br>Operations on Tensors

In [8]:
a = torch.tensor([1.0, 2.0])
b = torch.tensor([3.0, 4.0])

print('Element Wise Addition of a & b: \n', a + b)
print('Matrix Multiplication of a & b: \n', torch.matmul(a.view(2, 1), b.view(1, 2)))

Element Wise Addition of a & b: 
 tensor([4., 6.])
Matrix Multiplication of a & b: 
 tensor([[3., 4.],
        [6., 8.]])


## <br> Reshaping and Transposing Tensors

In [9]:
t = torch.tensor([[1, 2, 3, 4],
                 [5, 6, 7, 8],
                 [9, 10, 11, 12]])

print("Reshaping")
print(t.reshape(6, 2))

print("\nResizing")
print(t.view(2, 6))

print("\nTransposing")
print(t.transpose(0, 1))

Reshaping
tensor([[ 1,  2],
        [ 3,  4],
        [ 5,  6],
        [ 7,  8],
        [ 9, 10],
        [11, 12]])

Resizing
tensor([[ 1,  2,  3,  4,  5,  6],
        [ 7,  8,  9, 10, 11, 12]])

Transposing
tensor([[ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11],
        [ 4,  8, 12]])


## <br>Autograd & Computational Graphs

### Why do we need it?
Training a neural network means adjusting thousands of weights to minimize a loss. To know *which direction* to adjust each weight, you need its **gradient** (the derivative of the loss with respect to that weight). Computing this by hand for a large network is impossible — `autograd` does it automatically.

---

### How it works — 4 lines

```python
x = torch.tensor(2.0, requires_grad=True)  # 1. create tracked tensor
y = x ** 2                                  # 2. define a function (y = x²)
y.backward()                                # 3. compute gradient
print(x.grad)                               # 4. read result → tensor(4.)
```

| Line | What happens |
|---|---|
| `requires_grad=True` | Tells PyTorch: *watch this tensor, I'll need its gradient* |
| `y = x ** 2` | Records the operation in a graph: `x → (²) → y` |
| `y.backward()` | Walks the graph backwards, applies dy/dx = 2x → at x=2, result is **4.0** |
| `x.grad` | Stores that result — the slope of y at x=2 |

---

### Real-world meaning
In training: `x` = a model weight, `y` = the loss.  
`x.grad` tells the optimizer: *"nudge this weight by this much to reduce the loss."*  
This is the core of how every neural network learns.

In [10]:
import torch

x = torch.tensor(3.0, requires_grad=True)  
y = x ** 2  
y.backward()

print(f"x     = {x}")         # 2.0
print(f"y     = {y}")         # 9.0  (x²)
print(f"dy/dx = {x.grad}")    # 6.0  (2x at x=3)

x     = 3.0
y     = 9.0
dy/dx = 6.0


## <br>Building Neural Networks in PyTorch
PyTorch dynamically creates a computational graph that tracks operations and gradients for backpropagation.

![PyTorch](pytorch-workflow.webp)

In PyTorch, neural networks are built using the torch.nn module where:
- nn.Linear(in_features, out_features) defines a fully connected (dense) layer.
- Activation functions like torch.relu, torch.sigmoid or torch.softmax are applied between layers.
- forward() method defines how data moves through the network.
- To build a neural network in PyTorch, we create a class that inherits from torch.nn.Module and defines its layers and forward pass.


To build a neural network in PyTorch, we create a class that inherits from torch.nn.Module and defines its layers and forward pass.

In [11]:
import torch
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(10, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x
    
model = NeuralNetwork()
model


NeuralNetwork(
  (fc1): Linear(in_features=10, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=1, bias=True)
)

### Understanding the Neural Network

**`nn.Linear`** is a fully connected layer — every input connects to every output. It computes `output = input × Wᵀ + b`, where weights **W** and bias **b** are learned during training.

**`fc1`, `fc2`, `fc3`** — `fc` stands for *fully connected*. The numbers are just ordering. You could name them anything; PyTorch doesn't care.

**`Linear(in, out)`** — first number = features coming in, second = neurons in this layer.

| Layer | Shape | Role |
|---|---|---|
| `fc1 = Linear(10, 16)` | 10 → 16 | expand: find feature combinations |
| `fc2 = Linear(16, 8)` | 16 → 8 | compress: higher-level patterns |
| `fc3 = Linear(8, 1)` | 8 → 1 | output: a single prediction |

**The `forward()` lines connect the layers** — there is no separate "connect" call. Passing `x` through each layer sequentially *is* the connection:

```python
x = torch.relu(self.fc1(x))    # 10 → 16, zero out negatives
x = torch.relu(self.fc2(x))    # 16 → 8,  zero out negatives
x = torch.sigmoid(self.fc3(x)) # 8  → 1,  squash to 0–1 probability
```

> `nn.Sequential(nn.Linear(10,16), nn.ReLU(), ...)` is the shorthand equivalent.

**`x` is the data tensor flowing through the network.** It gets overwritten at each step — not accumulated, but *re-encoded* into a new representation:

```
x enters  →  10 raw features
after fc1 →  16 learned combinations   (shape: batch × 16)
after fc2 →  8  higher abstractions    (shape: batch × 8)
after fc3 →  1  probability 0–1        (shape: batch × 1)
```

**Network diagram** — expand then compress, a classic binary classifier shape:

```
  Input        fc1 + ReLU     fc2 + ReLU    fc3 + Sigmoid    Output
 (10 nodes)   (16 nodes)      (8 nodes)      (1 node)

    ●
    ●          ● ● 
    ●          ● ●         ● ●
    ●          ● ●         ● ●
    ●  ──────► ● ● ──────► ● ● ──────►  ●  ──►  [0.0 – 1.0]
    ●          ● ●         ● ●
    ●          ● ●        
    ●          ● ●
    ●          ● ●
    ●
```
*Every node on the left connects to every node on the right (fully connected).*

## <br>Define Loss Function and Optimizer
Once we define our model, we need to specify:

- A loss function to measure the error.
- An optimizer to update the weights based on computed gradients.

We use nn.BCELoss() for binary cross-entropy loss and used optim.Adam() for Adam optimizer to combine the benefits of momentum and adaptive learning rates.

In [ ]:
model = NeuralNetwork()
criterion = nn.BCELoss()  
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

<generator object Module.parameters at 0x11bfc5eb0>


## <br>Train the Model

The training involves generating dummy data (100 samples, each with 10 features). After this we run a training loop where we:

- optimizer.zero_grad() clears the accumulated gradients from the previous step.
- Forward Pass (model(inputs)) passes inputs through the model to generate predictions.
- Loss Computation (criterion(outputs, targets)) computes the difference between predictions and actual labels.
- Backpropagation (loss.backward()) computes gradients for all weights.
- Optimizer Step (optimizer.step()) updates the weights based on the computed gradients.

In [57]:
torch.manual_seed(42)

inputs = torch.randn((100, 10))
targets = torch.randint(0, 2, (100, 1)).float()
epochs = 50

# print("inputs: ", inputs)
# print("targets: ", targets)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [5/50], Loss: 0.6681
Epoch [10/50], Loss: 0.6681
Epoch [15/50], Loss: 0.6681
Epoch [20/50], Loss: 0.6681
Epoch [25/50], Loss: 0.6680
Epoch [30/50], Loss: 0.6680
Epoch [35/50], Loss: 0.6680
Epoch [40/50], Loss: 0.6680
Epoch [45/50], Loss: 0.6680
Epoch [50/50], Loss: 0.6680


## <br>PyTorch vs. TensorFlow

Let's see a quick difference between PyTorch and TensorFlow:

| Feature | PyTorch | TensorFlow |
|---|---|---|
| Computational Graph | Dynamic | Static (TF 1.x), Dynamic (TF 2.0) |
| Ease of Use | Pythonic, easy to debug | Steeper learning curve |
| Performance | Fast with eager execution | Optimized for large-scale deployment |
| Deployment | TorchScript & ONNX | TensorFlow Serving & TensorFlow Lite |
| Popularity in Research | Widely used | Also widely used but more in production |

### Applications

- **Computer Vision** — image classification, object detection, and segmentation using CNNs and Transformers (e.g., ViT).
- **Natural Language Processing (NLP)** — transformers, RNNs, and LSTMs for text generation and sentiment analysis.
- **Reinforcement Learning** — Deep Q-Networks (DQN), Policy Gradient Methods, and Actor-Critic Algorithms.